# 解压并验证完整脉冲数据

本 Notebook 用于把从GitHub下载的 `data/spike.zip` 安全解压到 `data/spike`。

流程：

1. 检查ZIP CRC、成员列表、路径穿越、符号链接和未压缩总大小。
2. 解压到临时目录 `data/.spike-extracting`。
3. 验证完整7700个样本的shape、dtype、二值范围、manifest和参考脉冲。
4. 验证通过后再原子移动到 `data/spike`。

默认不会覆盖已有的非空 `data/spike`。确需重建时设置环境变量：

```bash
export STEMNIST_OVERWRITE=1
```

In [ ]:
# 1. 导入依赖并定位项目
from __future__ import annotations

import csv
import json
import os
import shutil
import stat
from collections import Counter
from hashlib import sha256
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

import numpy as np


def find_project_root():
    """通过环境变量或当前目录向上寻找data/spike.zip。"""
    override = os.environ.get("STEMNIST_PROJECT_ROOT")
    start = Path(override).expanduser() if override else Path.cwd()
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (candidate / "data" / "spike.zip").is_file():
            return candidate

    raise FileNotFoundError(
        "未找到data/spike.zip。请在项目目录中启动Jupyter，或设置"
        "STEMNIST_PROJECT_ROOT。"
    )


def environment_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"环境变量{name}必须是布尔值，实际为{value!r}")


PROJECT_ROOT = find_project_root()
ARCHIVE_PATH = PROJECT_ROOT / "data" / "spike.zip"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "spike"
EXTRACT_ROOT = PROJECT_ROOT / "data" / ".spike-extracting"
OVERWRITE = environment_flag("STEMNIST_OVERWRITE", default=False)

LABELS = tuple("ABCDEFGHIJKLMNOPQRSTUVWXYZ123456789")
SPLITS = ("train", "val", "test")
EXPECTED_COUNTS = {"train": 5390, "val": 1155, "test": 1155}
EXPECTED_PER_CLASS = {"train": 154, "val": 33, "test": 33}
EXPECTED_FRAME_SHAPE = (240, 16, 16)
MAX_UNCOMPRESSED_BYTES = 600 * 1024**2

REQUIRED_MEMBERS = {
    "spike/metadata.json",
    "spike/train/spike.npy",
    "spike/train/manifest.csv",
    "spike/val/spike.npy",
    "spike/val/manifest.csv",
    "spike/test/spike.npy",
    "spike/test/manifest.csv",
}

print(f"项目目录：{PROJECT_ROOT}")
print(f"压缩包：{ARCHIVE_PATH}")
print(f"输出目录：{OUTPUT_ROOT}")

In [ ]:
# 2. ZIP安全检查与流式解压
def sha256_file(path, block_size=1024 * 1024):
    digest = sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def validate_member(info):
    """拒绝绝对路径、父目录跳转、反斜杠、符号链接和加密成员。"""
    name = info.filename
    if "\\" in name:
        raise RuntimeError(f"ZIP成员使用反斜杠：{name}")

    path = PurePosixPath(name)
    if path.is_absolute() or ".." in path.parts:
        raise RuntimeError(f"ZIP成员路径不安全：{name}")
    if not path.parts or path.parts[0] != "spike":
        raise RuntimeError(f"ZIP成员不在spike目录：{name}")
    if info.flag_bits & 0x1:
        raise RuntimeError(f"不支持加密ZIP成员：{name}")

    unix_mode = (info.external_attr >> 16) & 0xFFFF
    if unix_mode and stat.S_ISLNK(unix_mode):
        raise RuntimeError(f"ZIP中不允许符号链接：{name}")
    return path


def inspect_archive(archive_path=ARCHIVE_PATH):
    if not Path(archive_path).is_file():
        raise FileNotFoundError(f"找不到压缩包：{archive_path}")

    with ZipFile(archive_path, "r") as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"ZIP CRC校验失败：{bad_member}")

        file_infos = [info for info in archive.infolist() if not info.is_dir()]
        paths = [validate_member(info) for info in file_infos]
        names = [path.as_posix() for path in paths]
        if len(names) != len(set(names)):
            raise RuntimeError("ZIP中存在重复成员名称")

        member_names = set(names)
        if member_names != REQUIRED_MEMBERS:
            raise RuntimeError(
                "ZIP成员不正确："
                f"缺失={sorted(REQUIRED_MEMBERS - member_names)}，"
                f"额外={sorted(member_names - REQUIRED_MEMBERS)}"
            )

        total_size = sum(info.file_size for info in file_infos)
        if total_size > MAX_UNCOMPRESSED_BYTES:
            raise RuntimeError(
                f"ZIP解压后大小异常：{total_size / 1024**2:.2f} MiB"
            )

    return {
        "sha256": sha256_file(archive_path),
        "compressed_bytes": Path(archive_path).stat().st_size,
        "uncompressed_bytes": total_size,
        "member_count": len(names),
    }


def safe_extract(archive_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=False)

    with ZipFile(archive_path, "r") as archive:
        for info in archive.infolist():
            if info.is_dir():
                continue
            relative_path = validate_member(info)
            target = destination.joinpath(*relative_path.parts).resolve()
            if destination not in target.parents:
                raise RuntimeError(f"解压目标越界：{info.filename}")

            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info, "r") as source, target.open("wb") as output:
                shutil.copyfileobj(source, output, length=1024 * 1024)

In [ ]:
# 3. 验证解压后的数据契约
def read_csv_rows(path):
    with Path(path).open("r", encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def validate_spike_directory(spike_root):
    spike_root = Path(spike_root)
    metadata = read_json(spike_root / "metadata.json")
    if metadata.get("complete") is not True:
        raise RuntimeError("metadata未标记完整")
    if metadata.get("dataset_scope") != "full":
        raise RuntimeError("压缩包不是完整数据集")

    all_sample_ids = []
    for split_name in SPLITS:
        spike = np.load(
            spike_root / split_name / "spike.npy",
            mmap_mode="r",
            allow_pickle=False,
        )
        expected_shape = (
            EXPECTED_COUNTS[split_name],
            *EXPECTED_FRAME_SHAPE,
        )
        if spike.shape != expected_shape or spike.dtype != np.uint8:
            raise RuntimeError(
                f"{split_name}格式错误：{spike.shape}, {spike.dtype}"
            )
        if not np.all((spike == 0) | (spike == 1)):
            raise RuntimeError(f"{split_name}包含非二值数据")

        rows = read_csv_rows(spike_root / split_name / "manifest.csv")
        if len(rows) != EXPECTED_COUNTS[split_name]:
            raise RuntimeError(f"{split_name} manifest行数错误")
        if [int(row["row_index"]) for row in rows] != list(
            range(EXPECTED_COUNTS[split_name])
        ):
            raise RuntimeError(f"{split_name} manifest行号错误")

        class_counts = Counter(row["label"] for row in rows)
        if set(class_counts) != set(LABELS):
            raise RuntimeError(f"{split_name}类别集合错误")
        if set(class_counts.values()) != {EXPECTED_PER_CLASS[split_name]}:
            raise RuntimeError(f"{split_name}类别数量错误")

        all_sample_ids.extend(row["sample_id"] for row in rows)
        del spike

    if len(all_sample_ids) != 7700 or len(set(all_sample_ids)) != 7700:
        raise RuntimeError("样本ID不完整或存在重复")

    train_rows = read_csv_rows(spike_root / "train" / "manifest.csv")
    at_row = next(row for row in train_rows if row["sample_id"] == "AT_A_1")
    train_spike = np.load(
        spike_root / "train" / "spike.npy",
        mmap_mode="r",
        allow_pickle=False,
    )
    active_bins = np.flatnonzero(
        train_spike[int(at_row["row_index"]), :, 9, 6]
    )
    if not np.array_equal(active_bins, np.array([201, 203, 204, 206, 208])):
        raise RuntimeError(f"AT_A_1参考脉冲不正确：{active_bins}")
    del train_spike

    return metadata

In [ ]:
# 4. 解压、验证并原子安装到data/spike
def validate_safe_data_path(path):
    path = Path(path).resolve()
    data_root = (PROJECT_ROOT / "data").resolve()
    if path == data_root or data_root not in path.parents:
        raise ValueError(f"目标必须位于项目data目录：{path}")


def prepare_spike_data(overwrite=OVERWRITE):
    validate_safe_data_path(OUTPUT_ROOT)
    validate_safe_data_path(EXTRACT_ROOT)
    archive_info = inspect_archive(ARCHIVE_PATH)

    if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()) and not overwrite:
        raise FileExistsError(
            f"输出目录已经存在：{OUTPUT_ROOT}。"
            "如需重建，请设置STEMNIST_OVERWRITE=1。"
        )

    if EXTRACT_ROOT.exists():
        if not overwrite:
            raise RuntimeError(
                f"存在上次未完成的临时目录：{EXTRACT_ROOT}。"
                "确认后设置STEMNIST_OVERWRITE=1重建。"
            )
        shutil.rmtree(EXTRACT_ROOT)

    try:
        safe_extract(ARCHIVE_PATH, EXTRACT_ROOT)
        extracted_spike = EXTRACT_ROOT / "spike"
        metadata = validate_spike_directory(extracted_spike)

        if OUTPUT_ROOT.exists():
            if any(OUTPUT_ROOT.iterdir()):
                shutil.rmtree(OUTPUT_ROOT)
            else:
                OUTPUT_ROOT.rmdir()
        extracted_spike.replace(OUTPUT_ROOT)
        EXTRACT_ROOT.rmdir()
    except Exception:
        shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
        raise

    print(f"解压完成：{OUTPUT_ROOT}")
    print(
        f"ZIP大小：{archive_info['compressed_bytes'] / 1024**2:.2f} MiB"
    )
    print(f"ZIP SHA-256：{archive_info['sha256']}")
    return {"archive": archive_info, "metadata": metadata}


PREPARE_RESULT = prepare_spike_data()
PREPARE_RESULT

In [ ]:
# 5. 最终快速确认
metadata = read_json(OUTPUT_ROOT / "metadata.json")
assert metadata["complete"] is True
assert metadata["dataset_scope"] == "full"

for split_name in SPLITS:
    spike = np.load(
        OUTPUT_ROOT / split_name / "spike.npy",
        mmap_mode="r",
        allow_pickle=False,
    )
    print(f"{split_name}: shape={spike.shape}, dtype={spike.dtype}")
    del spike

print("脉冲数据准备完成。")